In [1]:
import os
import re
from dotenv import load_dotenv
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Load Environment Variables
load_dotenv("../.env")

# 2. Text Preprocessing & Cleaning Function
def clean_text(raw_text):
    cleaned = re.sub(r'\n+', '\n', raw_text)
    cleaned = re.sub(r' +', ' ', cleaned)
    return cleaned.strip()

# Load raw markdown data
file_path = "../data/extracted_enrollment_manual.md"
with open(file_path, "r", encoding="utf-8") as f:
    raw_markdown = f.read()

cleaned_markdown = clean_text(raw_markdown)

# 3. Demonstration of Tokenization (Requirement for Checkpoint 1)
tokenizer = tiktoken.get_encoding("cl100k_base")
tokens = tokenizer.encode(cleaned_markdown)

print("--- TEXT PREPROCESSING & TOKENIZATION DEMO ---")
print(f"Raw Character Count: {len(raw_markdown)}")
print(f"Cleaned Character Count: {len(cleaned_markdown)}")
print(f"Total Token Count: {len(tokens)}")
print(f"Sample Tokens (First 10 IDs): {tokens[:10]}")
print(f"Decoded Sample: '{tokenizer.decode(tokens[:10])}'\n")

# 4. Chunking (Text Splitting)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_text(cleaned_markdown)
print(f"Total Document Chunks Generated: {len(chunks)}")
print(f"Sample Chunk 1 Preview:\n{chunks[0]}\n")

# 5. Embedding Generation Demo (Free Local HuggingFace Model)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
sample_vector = embeddings_model.embed_query(chunks[0])

print("--- EMBEDDING GENERATION DEMO ---")
print(f"Embedding Vector Dimension: {len(sample_vector)}")
print(f"Sample Vector Values (First 5 Dimensions): {sample_vector[:5]}")

--- TEXT PREPROCESSING & TOKENIZATION DEMO ---
Raw Character Count: 2864
Cleaned Character Count: 2852
Total Token Count: 600
Sample Tokens (First 10 IDs): [4521, 5874, 220, 16, 55023, 51, 278, 581, 263, 19294]
Decoded Sample: '--- Page 1 ---
Talibon Poly'

Total Document Chunks Generated: 7
Sample Chunk 1 Preview:
--- Page 1 ---
Talibon Polytechnic College Admission RAG
 Knowledge Base
A chatbot-ready knowledge base adapted from the official TPC Student Handbook. This version
keeps only information useful for a college admission RAG chatbot and removes unrelated
disciplinary and administrative sections.
--- Page 10 ---
9. Admission Eligibility
Graduates of recognized public/private senior high schools may apply regardless of age, gender,



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- EMBEDDING GENERATION DEMO ---
Embedding Vector Dimension: 384
Sample Vector Values (First 5 Dimensions): [-0.052596285939216614, -0.014701740816235542, 0.05165175348520279, -0.0401252917945385, -0.031108317896723747]


In [25]:
import os
from langchain_community.vectorstores import Chroma

# Fixes for Windows Jupyter deadlocks
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

print("1. Preparing ChromaDB configuration...")
persist_directory = "../chroma_db"

print("2. Generating and saving vectors (Please wait 10-20 seconds)...")
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings_model,
    persist_directory=persist_directory
)

print(f"3. Successfully saved {len(chunks)} document chunks to ChromaDB.")

1. Preparing ChromaDB configuration...
2. Generating and saving vectors (Please wait 10-20 seconds)...
3. Successfully saved 7 document chunks to ChromaDB.


In [26]:
from langchain_community.document_loaders import PyPDFLoader
import re

# 1. Load your school PDF from the data folder
pdf_path = "../data/TPC_Admission_RAG_Knowledge_Base.pdf"  
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Total Pages Loaded from PDF: {len(pages)}")

# 2. Filter for Enrollment-related sections or extract all text to expand your dataset
enrollment_keywords = ["enrollment", "registration", "admission", "tuition", "dropping", "add/drop", "prerequisite"]

extracted_chunks = []
for i, page in enumerate(pages):
    page_text = page.page_content
    # Check if the page contains enrollment info, or keep all pages to hit your 20-30 page target
    if any(keyword in page_text.lower() for keyword in enrollment_keywords):
        extracted_chunks.append(f"--- Page {i+1} ---\n" + page_text)

print(f"Filtered down to {len(extracted_chunks)} pages containing enrollment policies.")

# 3. Save extracted text into a new Markdown file for your RAG database
output_path = "../data/extracted_enrollment_manual.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(extracted_chunks))

print(f"Successfully saved extracted enrollment data to {output_path}")

Total Pages Loaded from PDF: 26
Filtered down to 13 pages containing enrollment policies.
Successfully saved extracted enrollment data to ../data/extracted_enrollment_manual.md


In [27]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Load your generated markdown file (update the filename if needed)
md_path = "../data/extracted_enrollment_manual.md"
loader = UnstructuredMarkdownLoader(md_path)
docs = loader.load()

# 2. Split the markdown text into semantic chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(splits)}")

# 3. Create embeddings and store in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="../chroma_db"
)
print("Vector database successfully initialized and persisted.")

Total chunks created: 8


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database successfully initialized and persisted.


In [28]:
# 4. Test the retriever with a sample query
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
query = "What are the requirements for enrollment?"
retrieved_docs = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:300])
    print("\n")

Query: What are the requirements for enrollment?

--- Result 1 ---
## 3. The Five-Step Enrollment Workflow
During the designated enrollment period, applicants must strictly follow the sequential workflow below. 
### Step 1: Document Verification (Registrar's Office - Window 1)
The applicant submits all required documents. The Admission Officer evaluates the complet


--- Result 2 ---
--- Page 16 ---
15. Foreign Student Requirements
Alien Certificate of Registration and CHED-certified supporting documents depending on citizenship
status.
--- Page 17 ---
16. Enrollment Policies
Enrollment is valid only during the official registration period. Students become officially enrolled
on




In [3]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Load and connect to your existing ChromaDB vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(
    persist_directory="../chroma_db",
    embedding_function=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 2. Set up the LLM using the current gemini-3.6-flash model
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# 3. Define the RAG prompt and chain
template = """Answer the question based strictly on the provided context below. If you don't know the answer, state that you cannot find it in the manual.

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test your complete RAG bot
question = "What are the requirements for enrollment?"
response = rag_chain.invoke(question)

print(f"Question: {question}\n")
print(f"Answer:\n{response}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\63916\OneDrive\Desktop\THESIS IT PANCIT\admission-rag-bot\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What are the requirements for enrollment?

Answer:
I cannot find it in the manual.
